# gRPC benchmark results

This notebook loads every CSV produced by `run-all.sh` in `output/` and compares the current kotlinx-rpc client, the legacy client, and the direct Swift baseline. Change `selectedFiles` below to focus on a subset of runs.

In [57]:
%use lets-plot

import java.io.File
import kotlin.math.max
import org.jetbrains.letsPlot.*
import org.jetbrains.letsPlot.facet.*
import org.jetbrains.letsPlot.geom.*
import org.jetbrains.letsPlot.label.*
import org.jetbrains.letsPlot.scale.*
import org.jetbrains.letsPlot.themes.*

In [58]:
data class BenchmarkResult(
    val run: String,
    val benchmark: String,
    val caseName: String,
    val implementation: String,
    val platform: String,
    val target: String,
    val calls: Long,
    val concurrency: Int,
    val requestBytes: Long,
    val responseBytes: Long,
    val callsPerSecond: Double,
    val applicationBytesPerSecond: Double,
    val latencyP50Us: Double,
    val latencyP99Us: Double,
    val requestMessages: Long?,
    val responseMessages: Long?,
    val messagesPerSecond: Double?,
    val timeToFirstResponseUs: Double?,
    val finalResponseLatencyUs: Double?,
)

fun File.readBenchmarkResults(): List<BenchmarkResult> {
    val lines = readLines().filter(String::isNotBlank)
    require(lines.isNotEmpty()) { "Benchmark file is empty: $this" }

    // run-all.sh emits fields without CSV quoting, so splitting on commas matches its output format.
    val header = lines.first().split(',')
    val columnIndex = header.withIndex().associate { (index, name) -> name to index }
    fun List<String>.value(name: String): String = get(columnIndex.getValue(name))
    fun List<String>.optionalValue(name: String): String? = columnIndex[name]?.let { get(it) }?.takeIf(String::isNotBlank)

    return lines.drop(1).mapIndexed { index, line ->
        val columns = line.split(',')
        require(columns.size == header.size) { "Malformed row ${index + 2} in $this" }
        BenchmarkResult(
            run = nameWithoutExtension.removePrefix("grpc-benchmarks-"),
            benchmark = columns.value("benchmark"),
            caseName = columns.value("case"),
            implementation = columns.value("implementation"),
            platform = columns.value("platform"),
            target = columns.value("target"),
            calls = columns.value("calls").toLong(),
            concurrency = columns.value("concurrency").toInt(),
            requestBytes = columns.value("request_bytes").toLong(),
            responseBytes = columns.value("response_bytes").toLong(),
            callsPerSecond = columns.value("calls_per_second").toDouble(),
            applicationBytesPerSecond = columns.value("application_bytes_per_second").toDouble(),
            latencyP50Us = columns.value("latency_p50_us").toDouble(),
            latencyP99Us = columns.value("latency_p99_us").toDouble(),
            requestMessages = columns.optionalValue("request_messages")?.toLong(),
            responseMessages = columns.optionalValue("response_messages")?.toLong(),
            messagesPerSecond = columns.optionalValue("messages_per_second")?.toDouble(),
            timeToFirstResponseUs = columns.optionalValue("time_to_first_response_us")?.toDouble(),
            finalResponseLatencyUs = columns.optionalValue("final_response_latency_us")?.toDouble(),
        )
    }
}

In [59]:
val outputDirectory = listOf(
    File("output"),
    File("tests/grpc-benchmarks/output"),
).firstOrNull(File::isDirectory)
    ?: error("Run the notebook from the repository root or tests/grpc-benchmarks")

val csvFiles = outputDirectory
    .listFiles { file -> file.isFile && file.extension.equals("csv", ignoreCase = true) }
    ?.sortedBy(File::getName)
    .orEmpty()
require(csvFiles.isNotEmpty()) { "No benchmark CSV files found in $outputDirectory" }

// By default every run is overlaid. For only the latest run, use: listOf(csvFiles.last())
val selectedFiles = csvFiles
val results = selectedFiles.flatMap { f -> f.readBenchmarkResults() }

println("Loaded ${results.size} measurements from ${selectedFiles.size} file(s):")
selectedFiles.forEach { println("  ${it.name}") }

Loaded 156 measurements from 1 file(s):
  grpc-benchmarks-20260916-203912.csv


In [60]:
fun BenchmarkResult.payloadKind(): String = when {
    requestBytes == responseBytes -> if (requestBytes == 0L) "empty" else "symmetric"
    requestBytes > responseBytes -> "upload"
    else -> "download"
}

fun BenchmarkResult.payloadFacet(): String = when {
    benchmark == "unary-concurrency-sweep" && requestBytes == 0L -> "empty"
    benchmark == "unary-concurrency-sweep" -> "1 KiB each way"
    else -> payloadKind()
}

fun BenchmarkResult.payloadOrder(): Int = when (max(requestBytes, responseBytes)) {
    in 0L..64L -> 0
    in 65L..1_024L -> 1
    in 1_025L..65_536L -> 2
    in 65_537L..1_048_576L -> 3
    else -> 4
}

fun plotData(rows: List<BenchmarkResult>): Map<String, List<*>> = mapOf(
    "run" to rows.map(BenchmarkResult::run),
    "series" to rows.map { "${it.implementation} · ${it.run}" },
    "implementation" to rows.map(BenchmarkResult::implementation),
    "benchmark" to rows.map(BenchmarkResult::benchmark),
    "case" to rows.map(BenchmarkResult::caseName),
    "concurrency" to rows.map(BenchmarkResult::concurrency),
    "payload" to rows.map { it.payloadFacet() },
    "payload_order" to rows.map { it.payloadOrder() },
    "calls_per_second" to rows.map(BenchmarkResult::callsPerSecond),
    "messages_per_second" to rows.map(BenchmarkResult::messagesPerSecond),
    "application_mb_per_second" to rows.map { it.applicationBytesPerSecond / 1_000_000.0 },
    "latency_p50_us" to rows.map(BenchmarkResult::latencyP50Us),
    "latency_p99_us" to rows.map(BenchmarkResult::latencyP99Us),
)

data class LatencyPoint(val result: BenchmarkResult, val percentile: String, val microseconds: Double)

fun latencyPlotData(rows: List<BenchmarkResult>): Map<String, List<*>> {
    val points = rows.flatMap { result ->
        listOf(
            LatencyPoint(result, "p50", result.latencyP50Us),
            LatencyPoint(result, "p99", result.latencyP99Us),
        )
    }
    return mapOf(
        "run" to points.map { it.result.run },
        "series" to points.map { "${it.result.implementation} · ${it.result.run} · ${it.percentile}" },
        "implementation" to points.map { it.result.implementation },
        "concurrency" to points.map { it.result.concurrency },
        "payload" to points.map { it.result.payloadFacet() },
        "payload_order" to points.map { it.result.payloadOrder() },
        "percentile" to points.map(LatencyPoint::percentile),
        "microseconds" to points.map(LatencyPoint::microseconds),
    )
}

## Concurrency sweep

Higher call throughput is better. Each facet separates empty calls from calls with a 1 KiB request and response.

In [61]:
val concurrencyRows = results
    .filter { it.benchmark == "unary-concurrency-sweep" }
    .sortedWith(compareBy(BenchmarkResult::run, BenchmarkResult::implementation, BenchmarkResult::requestBytes, BenchmarkResult::concurrency))

letsPlot(plotData(concurrencyRows)) +
    geomLine(size = 1.1) {
        x = "concurrency"; y = "calls_per_second"; color = "implementation"
        linetype = "run"; group = "series"
    } +
    geomPoint(size = 2.7) {
        x = "concurrency"; y = "calls_per_second"; color = "implementation"
    } +
    facetGrid(x = "payload") +
    labs(
        title = "Unary throughput by concurrency",
        x = "Concurrent calls",
        y = "Calls per second",
        color = "Implementation",
        linetype = "Run",
    ) +
    themeMinimal() +
    ggsize(1000, 460)

0 
 
 
 
 
 
 
 20 
 
 
 
 
 
 
 40 
 
 
 
 
 
 
 60 
 
 
 
 
 
 
 80 
 
 
 
 
 
 
 100 
 
 
 
 
 
 
 120 
 
 
 
 
 
 
 
 
 5,000 
 
 
 
 
 
 
 10,000 
 
 
 
 
 
 
 15,000 
 
 
 
 
 
 
 20,000 
 
 
 
 
 
 
 25,000 
 
 
 
 
 
 
 
 1 KiB each way 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 
 
 20 
 
 
 
 
 
 
 40 
 
 
 
 
 
 
 60 
 
 
 
 
 
 
 80 
 
 
 
 
 
 
 100 
 
 
 
 
 
 
 120 
 
 
 
 
 
 
 
 
 
 empty 
 
 
 
 
 
 Unary throughput by concurrency 
 
 
 
 
 Calls per second 
 
 
 
 
 Concurrent calls 
 
 
 
 
 
 
 
 
 Run 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 20260916-203912 
 
 
 
 
 
 
 
 
 
 
 
 
 Implementation 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 current 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 legacy 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 swift

In [62]:
letsPlot(latencyPlotData(concurrencyRows)) +
    geomLine(size = 1.0) {
        x = "concurrency"; y = "microseconds"; color = "implementation"
        linetype = "percentile"; group = "series"
    } +
    geomPoint(size = 2.5) {
        x = "concurrency"; y = "microseconds"; color = "implementation"
        shape = "percentile"
    } +
    facetGrid(x = "payload") +
    scaleYLog10() +
    labs(
        title = "Unary latency by concurrency",
        subtitle = "Logarithmic latency scale",
        x = "Concurrent calls",
        y = "Latency (µs)",
        color = "Implementation",
        linetype = "Percentile",
        shape = "Percentile",
    ) +
    themeMinimal() +
    ggsize(1000, 460)

0 
 
 
 
 
 
 
 20 
 
 
 
 
 
 
 40 
 
 
 
 
 
 
 60 
 
 
 
 
 
 
 80 
 
 
 
 
 
 
 100 
 
 
 
 
 
 
 120 
 
 
 
 
 
 
 
 
 1,000 
 
 
 
 
 
 
 10,000 
 
 
 
 
 
 
 100,000 
 
 
 
 
 
 
 
 1 KiB each way 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 
 
 20 
 
 
 
 
 
 
 40 
 
 
 
 
 
 
 60 
 
 
 
 
 
 
 80 
 
 
 
 
 
 
 100 
 
 
 
 
 
 
 120 
 
 
 
 
 
 
 
 
 
 empty 
 
 
 
 
 
 Unary latency by concurrency 
 
 
 
 
 Logarithmic latency scale 
 
 
 
 
 Latency (µs) 
 
 
 
 
 Concurrent calls 
 
 
 
 
 
 
 
 
 Percentile 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 p50 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 p99 
 
 
 
 
 
 
 
 
 
 
 
 
 Implementation 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 current 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 legacy 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 swift

## Payload sweep

The x-axis follows the largest request or response body. Application throughput uses decimal MB/s, matching the benchmark's human-readable output.

In [63]:
val payloadRows = results
    .filter { it.benchmark == "unary-payload-sweep" }
    .sortedWith(compareBy(BenchmarkResult::run, BenchmarkResult::implementation, { it.payloadFacet() },
        { it.payloadOrder() }))

val payloadBreaks = listOf(0, 1, 2, 3, 4)
val payloadLabels = listOf("64 B", "1 KiB", "64 KiB", "1 MiB", "~4 MiB")

letsPlot(plotData(payloadRows)) +
    geomLine(size = 1.1) {
        x = "payload_order"; y = "application_mb_per_second"; color = "implementation"
        linetype = "run"; group = "series"
    } +
    geomPoint(size = 2.7) {
        x = "payload_order"; y = "application_mb_per_second"; color = "implementation"
    } +
    facetGrid(x = "payload") +
    scaleXContinuous(breaks = payloadBreaks, labels = payloadLabels) +
    labs(
        title = "Application throughput by payload size",
        x = "Largest request or response body",
        y = "Application throughput (MB/s)",
        color = "Implementation",
        linetype = "Run",
    ) +
    themeMinimal() +
    ggsize(1100, 460)

64 B 
 
 
 
 
 
 
 1 KiB 
 
 
 
 
 
 
 64 KiB 
 
 
 
 
 
 
 1 MiB 
 
 
 
 
 
 
 ~4 MiB 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 
 
 200 
 
 
 
 
 
 
 400 
 
 
 
 
 
 
 600 
 
 
 
 
 
 
 800 
 
 
 
 
 
 
 1,000 
 
 
 
 
 
 
 1,200 
 
 
 
 
 
 
 1,400 
 
 
 
 
 
 
 1,600 
 
 
 
 
 
 
 1,800 
 
 
 
 
 
 
 
 download 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 64 B 
 
 
 
 
 
 
 1 KiB 
 
 
 
 
 
 
 64 KiB 
 
 
 
 
 
 
 1 MiB 
 
 
 
 
 
 
 ~4 MiB 
 
 
 
 
 
 
 
 
 
 symmetric 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 64 B 
 
 
 
 
 
 
 1 KiB 
 
 
 
 
 
 
 64 KiB 
 
 
 
 
 
 
 1 MiB 
 
 
 
 
 
 
 ~4 MiB 
 
 
 
 
 
 
 
 
 
 upload 
 
 
 
 
 
 Application throughput by payload size 
 
 
 
 
 Application throughput (MB/s) 
 
 
 
 
 Largest request or response body 
 
 
 
 
 
 
 
 
 Run 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 20260916-203912 
 
 
 
 
 
 
 
 
 
 
 
 
 Implementation 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 current 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 legacy 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 swift

In [64]:
letsPlot(latencyPlotData(payloadRows)) +
    geomLine(size = 1.0) {
        x = "payload_order"; y = "microseconds"; color = "implementation"
        linetype = "percentile"; group = "series"
    } +
    geomPoint(size = 2.5) {
        x = "payload_order"; y = "microseconds"; color = "implementation"
        shape = "percentile"
    } +
    facetGrid(x = "payload") +
    scaleXContinuous(breaks = payloadBreaks, labels = payloadLabels) +
    scaleYLog10() +
    labs(
        title = "Unary latency by payload size",
        subtitle = "Logarithmic latency scale",
        x = "Largest request or response body",
        y = "Latency (µs)",
        color = "Implementation",
        linetype = "Percentile",
        shape = "Percentile",
    ) +
    themeMinimal() +
    ggsize(1100, 460)

64 B 
 
 
 
 
 
 
 1 KiB 
 
 
 
 
 
 
 64 KiB 
 
 
 
 
 
 
 1 MiB 
 
 
 
 
 
 
 ~4 MiB 
 
 
 
 
 
 
 
 
 1,000 
 
 
 
 
 
 
 10,000 
 
 
 
 
 
 
 100,000 
 
 
 
 
 
 
 
 download 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 64 B 
 
 
 
 
 
 
 1 KiB 
 
 
 
 
 
 
 64 KiB 
 
 
 
 
 
 
 1 MiB 
 
 
 
 
 
 
 ~4 MiB 
 
 
 
 
 
 
 
 
 
 symmetric 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 64 B 
 
 
 
 
 
 
 1 KiB 
 
 
 
 
 
 
 64 KiB 
 
 
 
 
 
 
 1 MiB 
 
 
 
 
 
 
 ~4 MiB 
 
 
 
 
 
 
 
 
 
 upload 
 
 
 
 
 
 Unary latency by payload size 
 
 
 
 
 Logarithmic latency scale 
 
 
 
 
 Latency (µs) 
 
 
 
 
 Largest request or response body 
 
 
 
 
 
 
 
 
 Percentile 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 p50 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 p99 
 
 
 
 
 
 
 
 
 
 
 
 
 Implementation 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 current 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 legacy 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 swift

## Headline results

These two plots isolate the default empty unary-latency case and the default 1 KiB, concurrency-16 unary-throughput case.

In [65]:
val defaultLatencyRows = results.filter { it.benchmark == "unary-latency" && it.caseName == "default" }

letsPlot(latencyPlotData(defaultLatencyRows)) +
    geomPoint(size = 5.0) {
        x = "implementation"; y = "microseconds"; color = "implementation"; shape = "percentile"
    } +
    labs(
        title = "Default unary latency",
        x = "Implementation",
        y = "Latency (µs)",
        color = "Implementation",
        shape = "Percentile",
    ) +
    themeMinimal() +
    ggsize(760, 420)

current 
 
 
 
 
 
 
 legacy 
 
 
 
 
 
 
 swift 
 
 
 
 
 
 
 
 
 150 
 
 
 
 
 
 
 200 
 
 
 
 
 
 
 250 
 
 
 
 
 
 
 300 
 
 
 
 
 
 
 350 
 
 
 
 
 
 
 400 
 
 
 
 
 
 
 450 
 
 
 
 
 
 
 500 
 
 
 
 
 
 
 550 
 
 
 
 
 
 
 600 
 
 
 
 
 
 
 
 
 Default unary latency 
 
 
 
 
 Latency (µs) 
 
 
 
 
 Implementation 
 
 
 
 
 
 
 
 
 Implementation 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 current 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 legacy 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 swift 
 
 
 
 
 
 
 
 
 
 
 
 
 Percentile 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 p50 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 p99

In [66]:
val defaultThroughputRows = results.filter { it.benchmark == "unary-throughput" && it.caseName == "default" }

letsPlot(plotData(defaultThroughputRows)) +
    geomPoint(size = 5.0) {
        x = "implementation"; y = "calls_per_second"; color = "implementation"
    } +
    labs(
        title = "Default unary throughput",
        subtitle = "1 KiB request + 1 KiB response at concurrency 16",
        x = "Implementation",
        y = "Calls per second",
        color = "Implementation",
    ) +
    themeMinimal() +
    ggsize(760, 420)

current 
 
 
 
 
 
 
 legacy 
 
 
 
 
 
 
 swift 
 
 
 
 
 
 
 
 
 9,000 
 
 
 
 
 
 
 10,000 
 
 
 
 
 
 
 11,000 
 
 
 
 
 
 
 12,000 
 
 
 
 
 
 
 13,000 
 
 
 
 
 
 
 14,000 
 
 
 
 
 
 
 15,000 
 
 
 
 
 
 
 
 
 Default unary throughput 
 
 
 
 
 1 KiB request + 1 KiB response at concurrency 16 
 
 
 
 
 Calls per second 
 
 
 
 
 Implementation 
 
 
 
 
 
 
 
 
 Implementation 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 current 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 legacy 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 swift

## Streaming results

Streaming CSV rows include actual request/response message counts and total messages per second. Higher throughput is better; latency measurements retain the call-shape-specific meaning documented in the benchmark README.

In [67]:
val directionalStreamingRows = results.filter {
    it.benchmark == "server-streaming-throughput" || it.benchmark == "client-streaming-throughput"
}.sortedWith(compareBy(BenchmarkResult::run, BenchmarkResult::benchmark, BenchmarkResult::implementation, { it.payloadOrder() }))

letsPlot(plotData(directionalStreamingRows)) +
    geomLine(size = 1.1) {
        x = "payload_order"; y = "application_mb_per_second"; color = "implementation"
        linetype = "run"; group = "series"
    } +
    geomPoint(size = 2.7) {
        x = "payload_order"; y = "application_mb_per_second"; color = "implementation"
    } +
    facetGrid(x = "benchmark") +
    scaleXContinuous(breaks = payloadBreaks, labels = payloadLabels) +
    labs(title = "Directional streaming throughput", x = "Measured message size", y = "Application throughput (MB/s)", color = "Implementation", linetype = "Run") +
    themeMinimal() +
    ggsize(1100, 460)

1 KiB 
 
 
 
 
 
 
 64 KiB 
 
 
 
 
 
 
 1 MiB 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 
 
 500 
 
 
 
 
 
 
 1,000 
 
 
 
 
 
 
 1,500 
 
 
 
 
 
 
 2,000 
 
 
 
 
 
 
 
 client-streaming-throughput 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 1 KiB 
 
 
 
 
 
 
 64 KiB 
 
 
 
 
 
 
 1 MiB 
 
 
 
 
 
 
 
 
 
 server-streaming-throughput 
 
 
 
 
 
 Directional streaming throughput 
 
 
 
 
 Application throughput (MB/s) 
 
 
 
 
 Measured message size 
 
 
 
 
 
 
 
 
 Run 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 20260916-203912 
 
 
 
 
 
 
 
 
 
 
 
 
 Implementation 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 current 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 legacy 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 swift

In [68]:
val bidiRows = results.filter { it.benchmark == "bidi-ping-pong" || it.benchmark == "bidi-full-duplex" }

letsPlot(plotData(bidiRows)) +
    geomPoint(size = 4.0) {
        x = "case"; y = "messages_per_second"; color = "implementation"; shape = "run"
    } +
    facetGrid(x = "benchmark", scales = "free_x") +
    labs(title = "Bidirectional streaming message rate", x = "Case", y = "Total messages per second", color = "Implementation", shape = "Run") +
    themeMinimal() +
    ggsize(1100, 460)

balanced-1k 
 
 
 
 
 
 
 balanced-64k 
 
 
 
 
 
 
 upload-heavy 
 
 
 
 
 
 
 download-heavy 
 
 
 
 
 
 
 
 
 10,000 
 
 
 
 
 
 
 20,000 
 
 
 
 
 
 
 30,000 
 
 
 
 
 
 
 40,000 
 
 
 
 
 
 
 50,000 
 
 
 
 
 
 
 60,000 
 
 
 
 
 
 
 70,000 
 
 
 
 
 
 
 
 bidi-full-duplex 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 empty 
 
 
 
 
 
 
 1k 
 
 
 
 
 
 
 
 
 
 bidi-ping-pong 
 
 
 
 
 
 Bidirectional streaming message rate 
 
 
 
 
 Total messages per second 
 
 
 
 
 Case 
 
 
 
 
 
 
 
 
 Implementation 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 current 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 legacy 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 swift 
 
 
 
 
 
 
 
 
 
 
 
 
 Run 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 20260916-203912

In [69]:
val streamScalingRows = results.filter { it.benchmark == "stream-concurrency-sweep" }
    .sortedWith(compareBy(BenchmarkResult::run, BenchmarkResult::implementation, BenchmarkResult::concurrency))

letsPlot(plotData(streamScalingRows)) +
    geomLine(size = 1.1) {
        x = "concurrency"; y = "messages_per_second"; color = "implementation"; linetype = "run"; group = "series"
    } +
    geomPoint(size = 2.7) { x = "concurrency"; y = "messages_per_second"; color = "implementation" } +
    labs(title = "Server-streaming throughput by stream concurrency", x = "Concurrent streams", y = "Total messages per second", color = "Implementation", linetype = "Run") +
    themeMinimal() +
    ggsize(900, 460)

2 
 
 
 
 
 
 
 4 
 
 
 
 
 
 
 6 
 
 
 
 
 
 
 8 
 
 
 
 
 
 
 10 
 
 
 
 
 
 
 12 
 
 
 
 
 
 
 14 
 
 
 
 
 
 
 16 
 
 
 
 
 
 
 
 
 50,000 
 
 
 
 
 
 
 60,000 
 
 
 
 
 
 
 70,000 
 
 
 
 
 
 
 80,000 
 
 
 
 
 
 
 90,000 
 
 
 
 
 
 
 100,000 
 
 
 
 
 
 
 
 
 Server-streaming throughput by stream concurrency 
 
 
 
 
 Total messages per second 
 
 
 
 
 Concurrent streams 
 
 
 
 
 
 
 
 
 Run 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 20260916-203912 
 
 
 
 
 
 
 
 
 
 
 
 
 Implementation 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 current 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 legacy 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 swift

In [70]:
val overheadRows = results.filter { it.benchmark == "stream-message-overhead" }
    .sortedWith(compareBy(BenchmarkResult::run, BenchmarkResult::implementation, { it.payloadOrder() }))

letsPlot(plotData(overheadRows)) +
    geomLine(size = 1.1) {
        x = "payload_order"; y = "messages_per_second"; color = "implementation"; linetype = "run"; group = "series"
    } +
    geomPoint(size = 2.7) { x = "payload_order"; y = "messages_per_second"; color = "implementation" } +
    scaleXContinuous(breaks = payloadBreaks, labels = payloadLabels) +
    scaleYLog10() +
    labs(title = "Fixed-64-MiB upload message overhead", subtitle = "Logarithmic message-rate scale", x = "Message size", y = "Total messages per second", color = "Implementation", linetype = "Run") +
    themeMinimal() +
    ggsize(900, 460)

1 KiB 
 
 
 
 
 
 
 64 KiB 
 
 
 
 
 
 
 1 MiB 
 
 
 
 
 
 
 ~4 MiB 
 
 
 
 
 
 
 
 
 316 
 
 
 
 
 
 
 1,000 
 
 
 
 
 
 
 3,162 
 
 
 
 
 
 
 10,000 
 
 
 
 
 
 
 31,623 
 
 
 
 
 
 
 
 
 Fixed-64-MiB upload message overhead 
 
 
 
 
 Logarithmic message-rate scale 
 
 
 
 
 Total messages per second 
 
 
 
 
 Message size 
 
 
 
 
 
 
 
 
 Run 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 20260916-203912 
 
 
 
 
 
 
 
 
 
 
 
 
 Implementation 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 current 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 legacy 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 swift